<img src="images/logo.png" width="15%" height="15%">

# **LangChain Essentials**

### **2. Messages**

LangChain accepts standard Python dictionaries to structure messages, such as `{"role": "user", "content": "hello"}`.

For maximum compatibility across different providers, LangChain accepts multiple aliases for each role:
- system / developer
- user / human
- assistant / ai
- tool / function


For more advanced control, LangChain provides explicit, powerful classes within `langchain_core.messages`:
- `SystemMessage`: Defines the core behavior, constraints, or persona for the LLM.
- `HumanMessage`: Represents user input. (When you pass a raw string to .invoke(), LangChain automatically converts it to this object type).
- `AIMessage`: The standardized object returned by the .invoke() method containing the model's response.
- `ToolMessage`: Used to return data from an executed tool back to the model.

These classes make it very easy to inspect metadata, extract text or tool calls, and display the conversation history nicely, as we will see below.

In [1]:
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()
llm = ChatMistralAI(model_name="mistral-small-latest")

In [2]:
from langchain_core.messages import HumanMessage

# We pass a HumanMessage to the model and receive an AIMessage in return
response = llm.invoke([HumanMessage(content="Hello World!")])
response

AIMessage(content="Hello, World! 🌍✨\n\nIt's great to see you here. How can I assist you today? Whether you need help with coding, answers to questions, or just want to chat, feel free to ask! 😊", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 18, 'total_tokens': 70, 'completion_tokens': 52, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019edbe8-1aaf-7c81-be97-4afdbd7cf505-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 52, 'total_tokens': 70})

In [3]:
# LangChain makes it very easy to extract the content from a message - Forget about response.choices[0].message.content!
response.content

"Hello, World! 🌍✨\n\nIt's great to see you here. How can I assist you today? Whether you need help with coding, answers to questions, or just want to chat, feel free to ask! 😊"

In [4]:
# The same goes with tool calls - There is none at the moment as we haven't provided tools
response.tool_calls

[]

In [5]:
response.usage_metadata

{'input_tokens': 18, 'output_tokens': 52, 'total_tokens': 70}

In [6]:
# All messages can be pretty_printed - Very helpful for easily displaying your conversation
response.pretty_print()

================================== Ai Message ==================================

Hello, World! 🌍✨

It's great to see you here. How can I assist you today? Whether you need help with coding, answers to questions, or just want to chat, feel free to ask! 😊


In [8]:
from langchain_core.messages import SystemMessage

# Let's try again with multiple messages now
memory = [
    SystemMessage(content="You are a travel assistant"),
    HumanMessage(content="Hello!", name="Tom") # The name is optional
]

response = llm.invoke(memory)

# The response is an AIMessage and can directly be appended to the memory (no need for response.choices[0].message any more)
memory.append(response)

# All message classes have a pretty_print method
for msg in memory:
    if not isinstance(msg, SystemMessage):
        msg.pretty_print()

================================ Human Message =================================
Name: Tom

Hello!
================================== Ai Message ==================================

Hello! How can I assist you with your travel plans today? 😊


In [9]:
# Adding a couple of additional messages for the next section
memory.append(HumanMessage(content="What should I visit in Brno?", name="Tom"))
response = llm.invoke(memory)
memory.append(response)

for msg in memory:
    msg.pretty_print()

================================ System Message ================================

You are a travel assistant
================================ Human Message =================================
Name: Tom

Hello!
================================== Ai Message ==================================

Hello! How can I assist you with your travel plans today? 😊
================================ Human Message =================================
Name: Tom

What should I visit in Brno?
================================== Ai Message ==================================

Brno, the second-largest city in the Czech Republic, is a vibrant destination with a mix of history, culture, and modern attractions. Here are some must-visit places and experiences:

### **Historical & Cultural Attractions**
1. **Špilberk Castle (Hrad Špilberk)**
   - A historic castle with a dark past (once a prison) and a great museum.
   - Offers panoramic views of Brno from its tower.
   - *Tip:* Visit the **Casemates** (underground tunne

In [ ]:
from langchain_core.messages import trim_messages

# https://reference.langchain.com/python/langchain-core/messages/utils/trim_messages
trimmer = trim_messages(
    max_tokens=3,         # Number of tokens (or messages in our case) to keep - Can be lower due to 'start_on="human"'
    strategy="last",      # Keep the last 'max_tokens' tokens/messages
    token_counter=len,    # the len function allows to count messages instead of tokens
    include_system=False,  # Whether to keep the SystemMessage if there is one at index 0.
    start_on="human",     # Forces the first message in the trimmed list to be of human type
)

# You can invoke the trimmer on your history list
trimmed_history = trimmer.invoke(memory)

[HumanMessage(content='What should I visit in Brno?', additional_kwargs={}, response_metadata={}, name='Tom'),
 AIMessage(content='Brno, the second-largest city in the Czech Republic, is a vibrant destination with a mix of history, culture, and modern attractions. Here are some must-visit places and experiences:\n\n### **Historical & Cultural Attractions**\n1. **Špilberk Castle (Hrad Špilberk)**\n   - A historic castle with a dark past (once a prison) and a great museum.\n   - Offers panoramic views of Brno from its tower.\n   - *Tip:* Visit the **Casemates** (underground tunnels) for a spooky experience.\n\n2. **Villa Tugendhat (UNESCO World Heritage Site)**\n   - A masterpiece of modernist architecture by Mies van der Rohe.\n   - *Book tickets in advance!* (Guided tours only.)\n\n3. **Brno Underground Labyrinth (Podzemí Brna)**\n   - Explore medieval cellars and tunnels beneath the city.\n   - *Highly recommended:* The **"Labyrinth of the Second Resistance"** tour.\n\n4. **Brno Ossua

In [15]:
memory.append(HumanMessage(content="Thanks"))

# A chain allows to create a sequential LLM workflow
chain = trimmer | llm

# Like any runnable, a chain has an invoke method
chain.invoke(memory)

AIMessage(content="You're welcome! I hope you have an amazing time exploring Brno—it's a fantastic city with so much to offer. If you need any more tips (like hidden gems, best cafés, or transport advice), just let me know. Enjoy your trip! 😊🍻🏰", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 916, 'total_tokens': 981, 'completion_tokens': 65, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019edbf4-1167-7091-823f-eecf125a7a8c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 916, 'output_tokens': 65, 'total_tokens': 981})

In [ ]:
from langchain_core.output_parsers import StrOutputParser

full_chain = chain | StrOutputParser() # Same as full_chain = trimmer | llm | StrOutputParser()
full_chain.invoke(memory)